> ⚠️ **机器翻译提示**：本 notebook 由机器翻译生成，未经人工校对，如发现翻译错误敬请谅解（代码与计算结果和越南语原版完全一致，只有文字说明是机器翻译）。

# 第02单元 — 抽样与样本量
第02单元专属 notebook —— 验证 PISA 抽样权重如何改变结果，以及基于统计检验力（power analysis）的样本量表（Cohen 1988，通过 `statsmodels` 计算）—— 这张表没有对应的 PSPP 命令，只能在这里运行。

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",4)

RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
print("学校表：", truong.shape)

## 1. 三个层次：总体、抽样框、样本
在 PISA 2025 越南调查中：总体 = 越南全体15岁学生；抽样框 = 越南教育部提供给 OECD 的学校名单；样本 = 实际参与的195所学校。

In [ ]:
print(f"实际样本：{truong.shape[0]} 所学校，共 {int(truong['n_hs'].sum())} 名学生")

## 2. 抽样权重如何改变结果 —— 真实验证
由于 PISA 按「与学校规模成比例的概率」进行整群抽样，各学校在分析中的「权重」并不相等。

In [ ]:
# 不加权：每所学校计为1
mean_unweighted = truong["EDULEAD"].mean()

# 加权：WEIGHT BY W_NRASCHBWT（相当于 SPSS 的 Data > Weight Cases）
valid = truong.dropna(subset=["EDULEAD"])
mean_weighted = np.average(valid["EDULEAD"], weights=valid["W_NRASCHBWT"])
n_eff = valid["W_NRASCHBWT"].sum()

print(f"EDULEAD 不加权平均值 = {mean_unweighted:.4f}（N={len(valid)}所学校，每校计为1）")
print(f"EDULEAD 加权平均值   = {mean_weighted:.4f}（有效N={n_eff:.1f}）")

#### 📤 实际输出
不加权 = 0.7653（N=195）。加权 = 0.7874（有效N=7284.6 —— 即权重总和，反映真实学生规模）。此处差异较小（0.02），因为 EDULEAD 与学校规模相关性不强，但对于与规模强相关的变量，忽略权重可能导致结论明显偏差。

## 3. 最小预试样本量（Julious, 2005）
n_pilot ≥ 每组12个观测值 —— 低于此阈值，估计的标准差波动过大，不适合作为正式样本量计算的输入。

In [ ]:
print("Julious (2005) 建议的最小预试样本量：每组12个观测值")

## 4. 基于检验力分析的正式样本量表（Cohen, 1988）
近似公式：n ≈ (z_α/2 + z_β)² × 2 / d² —— 用 `statsmodels.stats.power.TTestIndPower` 计算更精确（使用非中心t分布而非z近似）。

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"小"), (0.5,"中"), (0.8,"大")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
table = pd.DataFrame(rows, columns=["效应量 d", "等级", "每组样本量"])
table

#### 📤 实际输出
d=0.2→约394/组，d=0.5→约64/组，d=0.8→约26/组（α=0.05，power=0.80）。✅ 与第02、04单元中的表格一致。

## 5. 可视化：所需样本量随效应量增大而快速下降

In [ ]:
d_range = np.linspace(0.1, 1.0, 30)
n_range = [analysis.solve_power(effect_size=d, alpha=0.05, power=0.80, alternative="two-sided") for d in d_range]
fig, ax = plt.subplots()
ax.plot(d_range, n_range)
ax.set_xlabel("效应量 d"); ax.set_ylabel("每组所需样本量")
ax.set_title("所需样本量随效应量增大而快速下降（α=0.05，power=0.80）")
plt.show()